# IELTS Speaking Video Maker

**Before starting:** Runtime -> Change runtime type -> **T4 GPU**

| Cell | Action | Frequency |
|---|---|---|
| Cell 1 | Install | First time only |
| Cell 2 | Run app UI, click button | Every time |

---

### 🔑 Gemini API Key Setup (one-time)

This notebook reads your API key from **Colab Secrets** — your key is never stored in the notebook file.

1. Left sidebar → click the **🔑 (Secrets)** icon
2. Click **+ Add new secret**
3. Name: `GEMINI_API_KEY`
4. Value: paste your key from [aistudio.google.com](https://aistudio.google.com)
5. Toggle **Notebook access** to ON

Once saved, you never need to enter it again.


In [ ]:
from google.colab import userdata
userdata.get('IELTSGeminiAPIKey')

In [ ]:
# @title Setup - run once (~10 min)
import os, subprocess, urllib.request, zipfile

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr: print(r.stderr[-300:])

print('1/6 System packages...')
sh('apt-get update -q && apt-get install -y build-essential python3-dev python-is-python3 -q')
print('2/6 pip upgrade...')
sh('pip install --upgrade pip setuptools wheel -q')
print('3/6 OpenVoice...')
if not os.path.exists('/content/OpenVoice'):
    sh('git clone https://github.com/myshell-ai/OpenVoice.git /content/OpenVoice -q')
os.chdir('/content/OpenVoice')
sh('pip install -e . -q')
sh('pip install faster-whisper whisper-timestamped -q')
print('4/6 MeloTTS...')
sh('pip install git+https://github.com/myshell-ai/MeloTTS.git -q')
sh('pip install unidic -q && python -m unidic download')
print('5/6 Other packages...')
sh('pip install google-generativeai gTTS moviepy Pillow ipywidgets -q')
print('6/6 OpenVoice model download...')
ckpt_dir = '/content/OpenVoice/checkpoints_v2'
if not os.path.exists(ckpt_dir):
    url = 'https://myshell-public-repo-host.s3.amazonaws.com/openvoice/checkpoints_v2_0417.zip'
    urllib.request.urlretrieve(url, '/content/ckpt.zip')
    with zipfile.ZipFile('/content/ckpt.zip') as z:
        z.extractall('/content/OpenVoice/')
    os.remove('/content/ckpt.zip')
else:
    print('   (already downloaded - skip)')
print()
print('All done! Now run Cell 2.')


In [ ]:
# @title IELTS Speaking Video Maker
import os, sys, json, tempfile, warnings
import torch, numpy as np
import ipywidgets as w
from IPython.display import display, clear_output, HTML
from PIL import Image, ImageDraw, ImageFont
warnings.filterwarnings('ignore')
sys.path.insert(0, '/content/OpenVoice')

QUESTIONS = {
    'Part 1 - Daily Life': [
        'Tell me about your hometown. What do you like most about it?',
        'Do you enjoy cooking? Why or why not?',
        'How do you usually spend your weekends?',
        'What kind of music do you like? Why?',
        'Do you prefer to study in the morning or at night?',
    ],
    'Part 1 - Hobbies': [
        'What hobbies do you have?',
        'Do you enjoy reading books? What kind?',
        'How often do you exercise?',
    ],
    'Part 2 - Experience': [
        'Describe a memorable trip you have taken.',
        'Describe a time when you helped someone.',
        'Describe an achievement you are proud of.',
    ],
    'Part 3 - Society': [
        'Do you think technology has improved education? In what ways?',
        'How important is it for young people to learn a second language?',
        'What are the advantages and disadvantages of working from home?',
    ],
}
SCORE_DESC = {
    5.0: 'Basic - simple sentences',
    5.5: 'Elementary - more natural flow',
    6.0: 'Intermediate - complex structures',
    6.5: 'Upper-intermediate - varied vocabulary',
    7.0: 'Advanced - idioms included',
    7.5: 'Proficient - near-native',
    8.0: 'Expert - native level',
}
STYLE_MAP = {
    'Dark Minimal': ((15,15,25),(196,149,106),(235,235,235),(150,150,160)),
    'Light Clean':  ((248,246,241),(70,90,160),(30,30,40),(120,120,130)),
    'Deep Blue':    ((10,30,60),(100,180,255),(230,240,255),(140,170,210)),
}

state = dict(models_loaded=False, ref_path=None, tone_converter=None,
             melo_model=None, speaker_ids=None, device=None, target_se=None)

# --- UI CSS ---
CSS_BLOCK = '<style>\n.il-title{font-family:sans-serif;font-size:20px;font-weight:600;color:#1a1a1a;margin:0 0 4px}\n.il-sub{font-family:sans-serif;font-size:13px;color:#888;margin-bottom:18px}\n.il-label{font-family:sans-serif;font-size:11px;font-weight:700;letter-spacing:.1em;text-transform:uppercase;color:#aaa;margin:14px 0 4px}\n.il-result{background:#eef4fb;border:1px solid #b8d4ee;border-radius:8px;padding:14px 18px;font-family:monospace;font-size:13px;line-height:1.8;white-space:pre-wrap;margin-top:8px}\n.il-tip{background:#f0faf4;border:1px solid #a8dbb8;border-radius:8px;padding:12px 16px;font-size:13px;color:#2d6a4a;margin-top:8px}\n.il-ok{color:#2d7a4a;font-weight:600;font-family:sans-serif}\n.il-err{color:#c0392b;font-weight:600;font-family:sans-serif}\n.il-warn{color:#c07a00;font-weight:600;font-family:sans-serif}\nhr.il{border:none;border-top:1px solid #eee;margin:14px 0}\n</style>'
display(HTML(CSS_BLOCK))
display(HTML('<div class="il-title">IELTS Speaking Video Maker</div>'
             '<div class="il-sub">Korean answer -> IELTS English -> My Voice MP4</div>'))

# Gemini API Key - load from Colab Secrets
from google.colab import userdata
try:
    _api_key = userdata.get('GEMINI_API_KEY')
    display(HTML('<span class="il-ok">✓ Gemini API Key loaded from Secrets</span>'))
except Exception:
    _api_key = None
    display(HTML('<span class="il-err">⚠ GEMINI_API_KEY not found in Secrets.<br>'
                 'Left sidebar → 🔑 Secrets → Add new secret → Name: GEMINI_API_KEY</span>'))

# Voice upload
display(HTML('<hr class="il"><div class="il-label">My voice recording (10sec+ MP3/WAV)</div>'))
w_upload = w.FileUpload(accept='.mp3,.wav,.m4a', multiple=False)
out_upload = w.Output()
display(w_upload, out_upload)

def on_upload(change):
    with out_upload:
        clear_output()
        if not w_upload.value: return
        fname = list(w_upload.value.keys())[0]
        ext = os.path.splitext(fname)[1] or '.wav'
        path = '/content/my_voice' + ext
        with open(path, 'wb') as f:
            f.write(w_upload.value[fname]['content'])
        state['ref_path'] = path
        state['models_loaded'] = False
        display(HTML('<span class="il-ok">Uploaded: ' + fname + '</span>'))
w_upload.observe(on_upload, names='value')

# Question select
display(HTML('<hr class="il"><div class="il-label">Question</div>'))
w_part = w.Dropdown(options=list(QUESTIONS.keys()), layout=w.Layout(width='240px'))
w_q = w.Dropdown(options=QUESTIONS[w_part.value], layout=w.Layout(width='460px'))
def on_part(change): w_q.options = QUESTIONS[change['new']]
w_part.observe(on_part, names='value')
display(w_part, w_q)

# Korean answer
display(HTML('<div class="il-label" style="margin-top:10px">Korean answer (your story)</div>'))
w_korean = w.Textarea(
    placeholder='Example: I am from Busan. I love the sea...',
    layout=w.Layout(width='460px', height='110px')
)
display(w_korean)

# Band score
display(HTML('<hr class="il"><div class="il-label">Target IELTS Band Score</div>'))
w_score = w.SelectionSlider(options=[5.0,5.5,6.0,6.5,7.0,7.5,8.0], value=6.5,
                             layout=w.Layout(width='320px'))
out_sc = w.Output()
def show_sc(c):
    with out_sc:
        clear_output()
        display(HTML('<span style="font-size:12px;color:#888">' + SCORE_DESC[c['new']] + '</span>'))
w_score.observe(show_sc, names='value')
display(w_score, out_sc)
with out_sc: display(HTML('<span style="font-size:12px;color:#888">' + SCORE_DESC[6.5] + '</span>'))

# Style
display(HTML('<div class="il-label" style="margin-top:10px">Video Background</div>'))
w_style = w.ToggleButtons(options=['Dark Minimal','Light Clean','Deep Blue'], value='Dark Minimal')
display(w_style)

display(HTML('<hr class="il">'))
btn = w.Button(description='Generate Video', button_style='primary',
               layout=w.Layout(width='200px', height='44px'))
out_main = w.Output()
display(btn, out_main)

# --- Core functions ---
def load_models(ref_path):
    from openvoice import se_extractor
    from openvoice.api import ToneColorConverter
    from melo.api import TTS as MeloTTS
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    ckpt = '/content/OpenVoice/checkpoints_v2/converter'
    tc = ToneColorConverter(ckpt + '/config.json', device=device)
    tc.load_ckpt(ckpt + '/checkpoint.pth')
    target_se, _ = se_extractor.get_se(ref_path, tc, vad=True)
    mm = MeloTTS(language='EN', device=device)
    state.update(models_loaded=True, tone_converter=tc, melo_model=mm,
                 speaker_ids=mm.hps.data.spk2id, device=device, target_se=target_se)

def call_gemini(question, korean, score, api_key):
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    model = genai.GenerativeModel('gemini-2.0-flash')
    prompt = (
        f'You are an IELTS speaking coach. Transform the Korean answer to natural English for Band {score}.\n'
        f'Band {score}: {SCORE_DESC[score]}\n'
        'Rules: natural conversational tone, 60-120w Part1 / 150-200w Part2 / 80-120w Part3, use discourse markers.\n'
        f'Question: {question}\nKorean: {korean}\n'
        'Respond ONLY raw JSON: {"english":"...","korean":"Korean translation","band_tips":"Band explanation"}'
    )
    raw = model.generate_content(prompt).text.strip()
    if '```' in raw:
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    return json.loads(raw[raw.find('{'):raw.rfind('}')+1])

def gen_voice(text, out_path):
    tc=state['tone_converter']; mm=state['melo_model']
    spk=state['speaker_ids']; dev=state['device']; tse=state['target_se']
    tmp=out_path.replace('.wav','_base.wav')
    mm.tts_to_file(text, spk['EN-US'], tmp, speed=1.0)
    src_se=torch.load('/content/OpenVoice/checkpoints_v2/base_speakers/ses/en-us.pth', map_location=dev)
    tc.convert(audio_src_path=tmp, src_se=src_se, tgt_se=tse, output_path=out_path, message='@MyShell')
    if os.path.exists(tmp): os.remove(tmp)

def make_frame(style, label, text, score):
    W,H=1280,720
    img=Image.new('RGB',(W,H)); draw=ImageDraw.Draw(img)
    bg,ac,fg,mu=STYLE_MAP.get(style,STYLE_MAP['Dark Minimal'])
    draw.rectangle([0,0,W,H],fill=bg); draw.rectangle([60,80,80,H-80],fill=ac)
    try:
        fB=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',18)
        fM=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',28)
        fS=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',14)
    except: fB=fM=fS=ImageFont.load_default()
    draw.text((110,90),label.upper(),fill=ac,font=fB)
    words=text.split(); lines,cur=[],[]
    for wd in words:
        cur.append(wd)
        if len(' '.join(cur))>52: lines.append(' '.join(cur[:-1])); cur=[wd]
    lines.append(' '.join(cur))
    y=140
    for line in lines[:6]: draw.text((110,y),line,fill=fg,font=fM); y+=44
    draw.text((W-230,H-50),f'IELTS Band {score}',fill=mu,font=fS)
    return np.array(img)

# --- Button handler ---
def on_click(b):
    with out_main:
        clear_output()
        api_key=_api_key; question=w_q.value
        korean=w_korean.value.strip(); score=w_score.value; style=w_style.value
        if not api_key:
            display(HTML('<span class="il-err">Gemini API Key not found. Check Secrets setup above.</span>')); return
        if not korean:
            display(HTML('<span class="il-err">Please enter your Korean answer.</span>')); return
        if not state['ref_path']:
            display(HTML('<span class="il-err">Please upload your voice recording.</span>')); return
        tmpdir=tempfile.mkdtemp()
        if not state['models_loaded']:
            display(HTML('<span class="il-warn">Loading models... (1-2 min first time)</span>'))
            try:
                load_models(state['ref_path'])
                display(HTML('<span class="il-ok">Models loaded</span>'))
            except Exception as e:
                display(HTML('<span class="il-err">Model load failed: ' + str(e) + '</span>')); return
        display(HTML('<span class="il-warn">Generating English script...</span>'))
        try:
            result=call_gemini(question,korean,score,api_key)
            eng=result['english']
            display(HTML('<div class="il-result">' + eng + '</div>'))
            display(HTML('<div class="il-tip">' + result['korean'] + '<br><br>' + result['band_tips'] + '</div>'))
        except Exception as e:
            display(HTML('<span class="il-err">Script failed: ' + str(e) + '</span>')); return
        display(HTML('<span class="il-warn">Generating voice (my voice)...</span>'))
        q_path=os.path.join(tmpdir,'q.wav'); a_path=os.path.join(tmpdir,'a.wav')
        try:
            gen_voice(question,q_path); gen_voice(eng,a_path)
            display(HTML('<span class="il-ok">Voice generation complete</span>'))
        except Exception as e:
            display(HTML('<span class="il-err">Voice failed: ' + str(e) + '</span>')); return
        display(HTML('<span class="il-warn">Compositing video...</span>'))
        from moviepy.editor import AudioFileClip, ImageClip, concatenate_videoclips
        out_mp4='/content/ielts_speaking.mp4'
        try:
            qf=make_frame(style,'Question',question,score)
            af=make_frame(style,'Answer',eng,score)
            qa=AudioFileClip(q_path); aa=AudioFileClip(a_path)
            qc=ImageClip(qf).set_duration(qa.duration+1.5).set_audio(qa)
            ac=ImageClip(af).set_duration(aa.duration+2.0).set_audio(aa)
            final=concatenate_videoclips([qc,ac],method='compose')
            final.write_videofile(out_mp4,fps=24,codec='libx264',audio_codec='aac',logger=None)
            qa.close(); aa.close(); final.close()
            display(HTML('<span class="il-ok">Video complete!</span>'))
        except Exception as e:
            display(HTML('<span class="il-err">Video failed: ' + str(e) + '</span>')); return
        from google.colab import files
        files.download(out_mp4)
        display(HTML('<span class="il-ok">Download started. Upload to YouTube Studio!</span>'))

btn.on_click(on_click)
